In [76]:
# Goal: Build an end‑to‑end agentic application that :
  # composes ≥2 third‑party MCP servers + your own MCP server/tool,
  # plans with an LLM (GroqCloud or Ollama),
  # handles errors,
  # logs every step,
  # and exposes a minimal Streamlit UI.#

#1. Pick servers
  # Browse: Github Repository
  # Choose at least two that match your idea (look for clear README and simple setup).

In [77]:
# PREAMBULE :
# 1) Créer le dossier projet
!mkdir -p /content/mcp-mini-project
!cd /content/mcp-mini-project

# 2) Créer un venv (optionnel, Colab n'utilisera pas vraiment ce venv)
!python -m venv .venv

# 3) Créer tous les fichiers vides nécessaires
!touch requirements.txt .env.example servers.json llm_backends.py orchestrator.py my_insights_server.py app.py logging_utils.py

# 4) Remplir requirements.txt avec les librairies du projet
!printf "python-dotenv>=1.0.1\nrequests>=2.32.3\nstreamlit>=1.37.0\npydantic>=2.8.2\nrich>=13.7.1\ntenacity>=8.5.0\nmcp[cli]>=1.2.0\n" > requirements.txt

# 5) Installer les dépendances
!pip install -r requirements.txt

# 6) Vérifier que les fichiers existent bien
!ls -la

Error: Command '['/content/.venv/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.
total 36
drwxr-xr-x 1 root root 4096 Aug 21 14:19 .
drwxr-xr-x 1 root root 4096 Aug 21 12:26 ..
-rw-r--r-- 1 root root    0 Aug 21 14:19 app.py
drwxr-xr-x 4 root root 4096 Aug 19 13:37 .config
-rw-r--r-- 1 root root    0 Aug 21 14:19 .env.example
-rw-r--r-- 1 root root    0 Aug 21 14:19 llm_backends.py
-rw-r--r-- 1 root root    0 Aug 21 14:19 logging_utils.py
drwxr-xr-x 2 root root 4096 Aug 21 14:19 mcp-mini-project
-rw-r--r-- 1 root root    0 Aug 21 14:19 my_insights_server.py
-rw-r--r-- 1 root root 3323 Aug 21 14:19 orchestrator.py
drwxr-xr-x 2 root root 4096 Aug 21 13:27 __pycache__
-rw-r--r-- 1 root root  117 Aug 21 14:19 requirements.txt
drwxr-xr-x 1 root root 4096 Aug 19 13:38 sample_data
-rw-r--r-- 1 root root    0 Aug 21 14:19 servers.json
drwxr-xr-x 5 root root 4096 Aug 21 14:19 .venv


In [79]:
# 1. Pick Servers (Third‑Party) — Composition Requirement
  # Browse: Github Repository
  # Choose at least two that match your idea (look for clear README and simple setup).

  # Action: Choose at least two community MCP servers (from the course GitHub list) that fit your idea. Typical picks by scenario:
      # Smart Data Scout: a web/search server + a files/CSV server.
      # Dev Assistant: a repo/git server + an issues server (Jira/GitHub).
      # Research Notebook: a scholar/arXiv server + a notes/markdown server.

      # Function of this step:
        # run these servers locally (via node, python, docker, or a provided binary)
        # and define each one in servers.json so myclient can spawn & connect over STDIO (the MCP default transport).

      # Caractéristics :
        # name → étiquette pour reconnaître le serveur.
        # cmd → quel langage/commande utiliser (ex. Python, Node).
        # args → quel fichier lancer et avec quels paramètres.
        # env → quelles variables externes (clés API, chemins, etc.) sont nécessaires.

[
  {
    "name": "search-server",
    "cmd": "node",
    "args": ["./third_party/search-mcp-server/index.js"],
    "env": {
      "SEARCH_API_KEY": "${SEARCH_API_KEY}"
    }
  },
  {
    "name": "csv-server",
    "cmd": "python",
    "args": ["./third_party/csv_mcp_server.py"],
    "env": {}
  }
]

#✅ Tip: Prefer repos with a clear README, simple install (one command), and listed tools/resources. You can swap servers later without changing the orchestrator.

[{'name': 'search-server',
  'cmd': 'node',
  'args': ['./third_party/search-mcp-server/index.js'],
  'env': {'SEARCH_API_KEY': '${SEARCH_API_KEY}'}},
 {'name': 'csv-server',
  'cmd': 'python',
  'args': ['./third_party/csv_mcp_server.py'],
  'env': {}}]

In [11]:
#2. Choose your LLM backend
  # GroqCloud: set `GROQ_API_KEY`; configure base URL/model per their docs.
  # Ollama: install, `ollama pull ` (e.g., `llama3`), then point your client to the local endpoint.

In [24]:
# Choose one backend: groq or ollama - Config & Reproducibility Requirements

LLM_BACKEND='groq'

# Groq configuration (OpenAI-compatible)
GROQ_API_KEY="Ma clé Groq_API"
GROQ_MODEL='llama-3.1-70b-versatile' # example; change to any available Groq model

# Optional timeouts (seconds)
LLM_TIMEOUT=60
MCP_TOOL_TIMEOUT=60

# Example third-party server secrets (used by servers.json via env interpolation)
SEARCH_API_KEY='replace_me'

# Faire une copie en .env (le fichier réellement utilisé par le code)
!cp .env.example .env

# Vérifier que le fichier a bien été créé
!cat .env

In [25]:
# Test de connexion à Groq
import os
import requests

# 1) Charger la clé API depuis le .env
# Comme on est dans Colab, on peut soit lire directement le .env, soit la définir à la main
os.environ["GROQ_API_KEY"] = "Ma clé Groq_API"  # ⚠️ Remplace par ta vraie clé Groq

# 2) Définir l'endpoint Groq (compatible OpenAI)
url = "https://api.groq.com/openai/v1/chat/completions"

# 3) Préparer la requête
headers = {
    "Authorization": f"Bearer {os.environ['GROQ_API_KEY']}",
    "Content-Type": "application/json"
}

payload = {
    "model": "llama-3.1-70b-versatile",  # tu peux choisir un autre modèle dispo chez Groq
    "messages": [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "Hello, who are you?"}
    ],
    "temperature": 0.2
}

# 4) Envoyer la requête
response = requests.post(url, headers=headers, json=payload)

# 5) Vérifier et afficher la réponse
if response.status_code == 200:
    data = response.json()
    print("✅ Connexion réussie à Groq !")
    print("Réponse du modèle :\n")
    print(data["choices"][0]["message"]["content"])
else:
    print("❌ Erreur :", response.status_code, response.text)

❌ Erreur : 400 {"error":{"message":"The model `llama-3.1-70b-versatile` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.","type":"invalid_request_error","code":"model_decommissioned"}}



In [27]:
# Install Dependencies — Reproducibility

!requirements.txt

!python-dotenv>=1.0.1
!requests>=2.32.3
!streamlit>=1.37.0
!pydantic>=2.8.2
!rich>=13.7.1
!tenacity>=8.5.0

# MCP SDK (Python)
!mcp[cli]>=1.2.0

!pip install -r requirements.txt

/bin/bash: line 1: requirements.txt: command not found
/bin/bash: line 1: python-dotenv: command not found
/bin/bash: line 1: requests: command not found
Usage: streamlit [OPTIONS] COMMAND [ARGS]...

  Try out a demo with:

      $ streamlit hello

  Or use the line below to run your own script:

      $ streamlit run your_script.py

Options:
  --log_level [error|warning|info|debug]
  --version                       Show the version and exit.
  --help                          Show this message and exit.

Commands:
  activate  Activate Streamlit by entering your email.
  cache     Manage the Streamlit cache.
  config    Manage Streamlit's config settings.
  docs      Show help in browser.
  hello     Runs the Hello World script.
  help      Print this help message.
  init      Initialize a new Streamlit project.
  run       Run a Python script, piping stderr to Streamlit.
  version   Print Streamlit's version number.
/bin/bash: line 1: pydantic: command not found
/bin/bash: line 1: rich

In [ ]:
# 3. Implement your MCP client orchestration
  # Discover tools across all connected servers.
  # Prompt the LLM with the user goal and available tools to decide the next tool + args.
  # Execute step-by-step, stream results, and re-prompt after each step using accumulated context.
  # On failure, summarize the error and adapt (retry, swap tool, or adjust params).

In [28]:
# 4. Implement LLM Backends — Planning Requirement (LLM‑Driven)
  # Purpose: Provide a single chat() function regardless of Groq or Ollama.

!llm_backends.py

/bin/bash: line 1: llm_backends.py: command not found


In [29]:
import os
import json
import requests
from typing import List, Dict, Any

In [31]:
def _env(name: str, default: str | None = None) -> str:
  v = os.getenv(name, default)
  if v is None:
    raise RuntimeError(f"Missing required env var: {name}")
    return v

In [33]:
class LLMBackend:
  def chat(self, messages: List[Dict[str, str]]) -> str:
    raise NotImplementedError

class GroqBackend(LLMBackend):
  def __init__(self):
    self.api_key = _env("GROQ_API_KEY")
    self.model = os.getenv("GROQ_MODEL", "llama-3.1-70b-versatile")
    self.timeout = int(os.getenv("LLM_TIMEOUT", "60"))
    self.base_url = "https://api.groq.com/openai/v1/chat/completions"

  def chat(self, messages: List[Dict[str, str]]) -> str:
    headers = {
      "Authorization": f"Bearer {self.api_key}",
      "Content-Type": "application/json",
    }
    payload = {
      "model": self.model,
      "messages": messages,
      "temperature": 0.2,
      "stream": False,
    }
    r = requests.post(self.base_url, headers=headers, data=json.dumps(payload), timeout=self.timeout)
    r.raise_for_status()
    data = r.json()

    return data["choices"][0]["message"]["content"]

class OllamaBackend(LLMBackend):
  def __init__(self):
    self.base = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
    self.model = os.getenv("OLLAMA_MODEL", "llama3")
    self.timeout = int(os.getenv("LLM_TIMEOUT", "60"))

  def chat(self, messages: List[Dict[str, str]]) -> str:
    url = f"{self.base}/api/chat"
    payload = {
      "model": self.model,
      "messages": messages,
      "stream": False,
    }
    r = requests.post(url, json=payload, timeout=self.timeout)
    r.raise_for_status()
    data = r.json()
    # Ollama returns an array of messages or a single object depending on version
    if isinstance(data, dict) and "message" in data:
      return data["message"]["content"]
    # Fallback for some versions that return a list of chunks
    if isinstance(data, list) and data:
      return "".join(chunk.get("message", {}).get("content", "") for chunk in data)
    raise RuntimeError("Unexpected Ollama response format")

def make_backend() -> LLMBackend:
  backend = os.getenv("LLM_BACKEND", "groq").lower()
  if backend == "groq":
    return GroqBackend()
  if backend == "ollama":
    return OllamaBackend()
  raise RuntimeError(f"Unsupported LLM_BACKEND: {backend}")

In [34]:
#5. Observability & Logs — Observability Requirement
  # Purpose: Log every tool call (inputs/outputs summarized, avoid secrets), plus high‑level planner messages.

!logging_utils.py

/bin/bash: line 1: logging_utils.py: command not found


In [36]:
from __future__ import annotations
import json, os, time
from typing import Any, Dict
from pathlib import Path

LOG_DIR = Path("logs"); LOG_DIR.mkdir(exist_ok=True)

class JsonlLogger:
  def __init__(self, name: str):
    ts = time.strftime("%Y%m%d-%H%M%S")
    self.path = LOG_DIR / f"{name}-{ts}.jsonl"

  def log(self, kind: str, payload: Dict[str, Any]):
    # redact common secret‑like keys
    redacted = {}
    for k, v in payload.items():
      if any(s in k.lower() for s in ["token", "key", "secret", "authorization"]):
        redacted[k] = "[REDACTED]"
      else:
        redacted[k] = v
    with self.path.open("a", encoding="utf-8") as f:
      f.write(json.dumps({"kind": kind, **redacted}) + "\n")

In [13]:
!pip install "mcp[cli]"
!pip install --upgrade mcp[cli]
from mcp.server import Server
from mcp.types import Tool, CallToolRequest, CallToolResult

In [20]:
!pip show mcp
!await server.run()

Name: mcp
Version: 1.13.0
Summary: Model Context Protocol SDK
Home-page: https://modelcontextprotocol.io
Author: Anthropic, PBC.
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: anyio, httpx, httpx-sse, jsonschema, pydantic, pydantic-settings, python-multipart, sse-starlette, starlette, uvicorn
Required-by: google-adk
/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `await server.run()'


In [28]:
!pip show mcp

Name: mcp
Version: 1.13.0
Summary: Model Context Protocol SDK
Home-page: https://modelcontextprotocol.io
Author: Anthropic, PBC.
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: anyio, httpx, httpx-sse, jsonschema, pydantic, pydantic-settings, python-multipart, sse-starlette, starlette, uvicorn
Required-by: google-adk


In [33]:
from mcp.server.stdio import stdio_server

async def amain():
    await stdio_server(server)

In [36]:
# 1) Aller dans ton projet
# %cd /content/mcp-mini-project # Removed redundant cd

# 2) Installer/mettre à jour MCP SDK (dernière version stable)
!pip install -U "mcp[cli]"

# 3) Créer le fichier my_insights_server.py avec la version corrigée
!cat > my_insights_server.py << 'PY'

import asyncio
from mcp.server import Server
from mcp.types import Tool, CallToolRequest, CallToolResult
from mcp.server.stdio import stdio_server   # ✅ bon import en v1.13.0

server = Server("insights-server")

@server.list_tools()
async def list_tools() -> list[Tool]:
    return [
        Tool(
            name="summarize_text",
            description="Summarize input text into up to 5 bullet points.",
            inputSchema={
                "type": "object",
                "properties": {"text": {"type": "string"}},
                "required": ["text"],
            },
        )
    ]

@server.call_tool()
async def call_tool(req: CallToolRequest) -> CallToolResult:
    if req.name == "summarize_text":
        text = (req.arguments or {}).get("text", "")
        bullets = [f"• {s.strip()}" for s in text.split(". ") if s.strip()][:5]
        return CallToolResult(content="\n".join(bullets) or "• (no content)")
    raise ValueError(f"Unknown tool: {req.name}")

async def amain():
    # ✅ nouvelle API en 1.13.0
    await stdio_server(server)

# Dans Colab, tu peux directement lancer :
# await amain()

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')


In [64]:
# 7) Orchestrator — Planning, Error Handling, Config
  # Purpose:
    # Spawn/connect to all servers in servers.json + your my_insights_server.py.
    # Discover tools/resources across servers.
    # Ask the LLM to plan the next tool call given the user goal and available tools.
    # Execute, stream results, handle failures with retries/fallbacks, and log everything.


In [40]:
servers_cfg = [
    {
        "name": "search-server",
        "cmd": "node",
        "args": ["./third_party/search-mcp-server/index.js"],
        "env": {"SEARCH_API_KEY": "ma_clef"}
    },
    {
        "name": "csv-server",
        "cmd": "python",
        "args": ["./third_party/csv_mcp_server.py"],
        "env": {}
    }
]

for s in servers_cfg:
    print("Serveur :", s["name"], "commande :", s["cmd"])

Serveur : search-server commande : node
Serveur : csv-server commande : python


In [42]:
import subprocess
import os

class ServerProc:
    def __init__(self, name: str, cmd: str, args: list[str], env: dict[str, str]):
        self.name = name
        self.cmd = cmd
        self.args = args
        self.env = {**os.environ, **env}
        self.proc: subprocess.Popen | None = None

    def start(self):
        self.proc = subprocess.Popen(
            [self.cmd, *self.args],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            env=self.env
        )
        if not self.proc or not self.proc.stdin or not self.proc.stdout:
            raise RuntimeError(f"❌ Impossible de démarrer le serveur {self.name}")
        print(f"✅ Serveur {self.name} lancé")
        return self.proc

    def stop(self):
        if self.proc and self.proc.poll() is None:
            self.proc.terminate()
            print(f"🛑 Serveur {self.name} arrêté")


In [44]:
from __future__ import annotations
# Start third‑party servers
procs: dict[str, ServerProc] = {}
for s in servers_cfg:
    name = s["name"]
    cmd = s["cmd"]
    args = s.get("args", [])
    env = s.get("env", {})

    # interpolation éventuelle des variables d’environnement
    env = {k: (os.path.expandvars(v) if isinstance(v, str) else v) for k, v in env.items()}

    sp = ServerProc(name, cmd, args, env)
    sp.start()

✅ Serveur search-server lancé
✅ Serveur csv-server lancé


In [47]:
# Connect to all
for name, sp in procs.items():
  console.print(f"[bold green]Connecting to {name}…[/]")
  sessions[name] = await connect_server(sp)

In [50]:
# Discover tools
catalog: List[Tuple[str, List[Dict[str, Any]]]] = []
sessions: dict[str, Any] = {} # Initialize the sessions dictionary

for name, sess in sessions.items():
    tools = await list_all_tools(sess)
    catalog.append((name, tools))
    logger.log("catalog", {"servers": [name for name in sessions]})

In [56]:
import asyncio
import json
import sys
from typing import List, Dict, Any, Tuple

# Assume these functions and variables are defined elsewhere in the notebook:
# build_planner_prompt, llm, logger, console, sessions, call_tool, timeout, procs, user_goal, catalog

async def run_goal(user_goal: str, catalog: List[Tuple[str, List[Dict[str, Any]]]], sessions: dict[str, Any], llm: Any, logger: Any, console: Any, procs: dict[str, Any], timeout: int):
    history: List[Dict[str, str]] = []
    try:
        for step in range(1, 8): # up to 7 steps
            prompt = build_planner_prompt(user_goal, catalog, history)
            plan_json = llm.chat(prompt)
            logger.log("plan", {"step": step, "raw": plan_json})
            try:
                plan = json.loads(plan_json)
                server_name = plan["server"]
                tool = plan["tool"]
                args = plan.get("args", {})
                rationale = plan.get("rationale", "")
            except (json.JSONDecodeError, KeyError, TypeError, ValueError) as e:
                # If parsing fails, ask the LLM to reformat
                history.append({"role": "assistant", "content": f"Plan parse error: {e}. Please return STRICT JSON only."})
                continue
            except Exception as e:
                # Catch any other unexpected errors during parsing
                history.append({"role": "assistant", "content": f"Unexpected parsing error: {e}. Please return STRICT JSON only."})
                continue

            if server_name not in sessions:
                history.append({"role": "assistant", "content": f"Planner selected unknown server '{server_name}'. Try again."})
                continue

            console.print(f"[bold]Step {step}[/]: {server_name}.{tool} -> args={args}")
            logger.log("tool_call", {"step": step, "server": server_name, "tool": tool, "args": args})

            try:
                # This function needs to be defined or imported elsewhere
                output = await call_tool(sessions[server_name], tool, args, timeout)
                logger.log("tool_result", {"step": step, "server": server_name, "tool": tool, "result_preview": output[:500]})
                # Feed result back to planner
                history.append({"role": "assistant", "content": f"Result from {server_name}.{tool}:\n{output[:2000]}"})
                # Heuristic: if the planner states "goal achieved", stop
                if "goal achieved" in output.lower() or "done" in output.lower():
                    break
            except Exception as e:
                logger.log("tool_error", {"step": step, "server": server_name, "tool": tool, "error": str(e)})
                # Error handling strategy: retry via history hint
                history.append({"role": "assistant", "content": f"Tool error: {e}. Consider retrying with adjusted parameters or another tool."})

        console.print("[green]\nRun complete. See logs/ for JSONL traces.\n[/]")

    finally:
        for sp in procs.values():
            sp.stop()

# To run this function in Colab, ensure all required arguments (user_goal, catalog, sessions, llm, logger, console, procs, timeout) are defined
# and then call it using:
# await run_goal(user_goal, catalog, sessions, llm, logger, console, procs, timeout)

In [65]:
!orchestrator.py

/bin/bash: line 1: orchestrator.py: command not found


In [62]:
# 8) Streamlit UI — Integration
  # Purpose: Provide a minimal UI to enter a goal, select backend, and watch steps/logs.

!app.py

/bin/bash: line 1: app.py: command not found


In [67]:
!pip install streamlit

import os, asyncio, streamlit as st
from dotenv import load_dotenv
from orchestrator import run_goal

load_dotenv(override=True)

st.set_page_config(page_title="MCP Agent Orchestrator", layout="centered")
st.title("🤖 MCP Agent Orchestrator")


backend = st.selectbox("LLM Backend", ["groq", "ollama"], index=0)
st.text_input("LLM_BACKEND (env)", value=backend, key="backend_env")

user_goal = st.text_area("User Goal", "Fetch a CSV from the web and produce a 5‑bullet summary")

col1, col2 = st.columns(2)
with col1:
  run_button = st.button("Run Plan")
  with col2:
    st.link_button("View Logs Folder", "#", help="Logs are written to ./logs as JSONL")

if run_button:
  os.environ["LLM_BACKEND"] = st.session_state["backend_env"].strip()
  with st.spinner("Running… check the terminal for live output."):
    # Streamlit + asyncio: run the orchestrator in the same thread
    asyncio.run(run_goal(user_goal))
  st.success("Done. Open the newest file in ./logs to inspect the step‑by‑step trace.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 93.3 MB/s eta 0:00:00


2025-08-21 13:27:56.028 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-21 13:27:56.029 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-21 13:27:56.149 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-08-21 13:27:56.150 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-21 13:27:56.152 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-21 13:27:56.154 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-21 13:27:56.155 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

#9. Functional Requirements Checklist — Where Each Is Implemented
    # Composition (≥2 third‑party + your server): servers.json + my_insights_server.py + orchestrator.py spawning/connecting all.
    # Planning (LLM chooses order): build_planner_prompt() + LLM call in run_goal() loop, not hard‑coded order.
    # Error handling: tenacity retry in call_tool(), planner receives error summaries to adapt.
    # Observability: logging_utils.JsonlLogger writes JSONL entries for catalog, plan, tool_call, tool_result, tool_error.
    # Config: .env governs LLM backend & timeouts; servers.json governs server processes; env interpolation for secrets.
    # Reproducibility: requirements.txt + one‑shot setup + python orchestrator.py "<goal>" or streamlit run app.py.

In [68]:
# Run It — Simple Startup (Clean Machine)
  # Step 1 - Aller dans le dossier du projet
!cd /content/mcp-mini-project

  # Step 2 - Vérifier que le fichier .env existe (sinon, copier l'exemple)
!test -f .env || cp .env.example .env

  # Step 3 - Exemple : lancer l’orchestrateur avec un objectif clair
!python orchestrator.py "Fetch a CSV from the web and produce a 5-bullet summary"

  # Step 4 - Lancer l’UI Streamlit (optionnel)
# Streamlit lance un serveur web, Colab n'affiche pas directement, mais tu peux utiliser ngrok ou colab-tunnel
# !streamlit run app.py

/bin/bash: line 1: cd: /content/mcp-mini-project: No such file or directory
cp: cannot stat '.env.example': No such file or directory


In [75]:
# Aller dans le dossier projet
!cd /content/mcp-mini-project

# SMART DATA SCOUT
!python orchestrator.py "Find a public CSV about inflation rates, clean headers if needed, and produce a 5-bullet executive summary"

# DEV ASSISTANT
!python orchestrator.py "List open issues labeled 'bug', pick one small ticket, and propose a minimal patch (diff) with a rationale"

# RESEARCH NOTEBOOK
!python orchestrator.py "Retrieve two 2024-2025 papers on retrieval-augmented generation, and output a citation-clean outline"

%cd /content/mcp-mini-project

!Smart Data Scout

!python orchestrator.py "Find a public CSV about inflation rates, clean headers if needed, and produce a 5-bullet executive summary"
"➝ But : utiliser un serveur de recherche + un serveur CSV + ton serveur custom pour résumer les données."

!Dev Assistant

/bin/bash: line 1: cd: /content/mcp-mini-project: No such file or directory
[Errno 2] No such file or directory: '/content/mcp-mini-project'
/content
/bin/bash: line 1: Smart: command not found
/bin/bash: line 1: Dev: command not found


#1. Pick servers
  # Browse: Github Repository
  # Choose at least two that match your idea (look for clear README and simple setup).
# 2. Choose your LLM backend
  # GroqCloud: set `GROQ_API_KEY`; configure base URL/model per their docs.
  # Ollama: install, `ollama pull ` (e.g., `llama3`), then point your client to the local endpoint.
# 3. Implement your MCP client orchestration
  # Discover tools across all connected servers.
  # Prompt the LLM with the user goal and available tools to decide the next tool + args.
  # Execute step-by-step, stream results, and re-prompt after each step using accumulated context.
On failure, summarize the error and adapt (retry, swap tool, or adjust params).
# 4. Integrate your pipeline in Streamlit
You can build different components with Streamlit.


# Mini-Project (Part 1): Integrate thied-part MCP servers to build powerful AI applications


👩‍🏫 👩🏿‍🏫 what you’ve learned in the course
# You created tools and agents, exposed them through an MCP server, and used an LLM-driven client to plan and execute via orchestration.
# You tried different agentic frameworks to build and coordinate tool-using agents.

But MCP is more powerful than that
# The power of MCP is the ecosystem:
  # people and companies now publish MCP servers that expose their tools.
  # You can build powerful apps by integrating existing MCP servers with your own, no need to recreate all the tools.

# Your task : Build an end-to-end agentic application that:
  # Integrates at least two third-party MCP servers from the community list:
    # Github Repository
    # Runs those servers locally and connects to them from an MCP client.
    # Uses an LLM to plan and orchestrate tool calls to achieve a clear user goal.
    # Works with either GroqCloud (hosted LLMs) or Ollama (local models).

# Example project ideas (you can think of your own)
  # Smart Data Scout: web/search server + files/CSV server + your “insights” tool to fetch, clean, and summarize data.
  # Dev Assistant: repo/git server + issues/Jira or GitHub server + your “lint/fix” tool to triage and propose a patch.
  # Research Notebook: scholar/arXiv server + notes/markdown server + your “citation-cleaner” tool to assemble a brief.

# Functional requirements
  # Composition: ≥ 2 third-party MCP servers plus your server in one flow.
  # Planning: the LLM chooses the order of tool calls (not fully hard-coded).
  # Error handling: handle timeouts/bad inputs with retries or fallback steps.
  # Observability: log each tool call (inputs/outputs summarized; no secrets).
  # Config: env-based setup for LLM backend (Groq or Ollama) and server endpoints.
  # Reproducibility: simple startup (short command sequence) on a clean machine.